In [1]:
import pandas as pd
import numpy as np

metadata = pd.read_csv("FRANZOSA_IBD_2019/FRANZOSA_IBD_2019/metadata.tsv", sep="\t")
mtb = pd.read_csv("FRANZOSA_IBD_2019/FRANZOSA_IBD_2019/mtb.tsv", sep="\t")

feature_cols = [c for c in mtb.columns if c != "Sample"]
X = metadata.merge(mtb, on="Sample")[feature_cols]
y = metadata["Study.Group"]

print("X shape:", X.shape)
print(y.value_counts())

X shape: (220, 8848)
Study.Group
CD         88
UC         76
Control    56
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_enc = le.fit_transform(y)

train_idx, test_idx = train_test_split(np.arange(len(y_enc)), test_size=0.2, stratify=y_enc, random_state=42)
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y_enc[train_idx], y_enc[test_idx]

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Classes:", le.classes_)

Train: (176, 8848) Test: (44, 8848)
Classes: ['CD' 'Control' 'UC']


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X_train_filled = X_train.fillna(0)
X_test_filled = X_test.fillna(0)

rf_metaml = RandomForestClassifier(n_estimators=500, max_depth=None, min_samples_split=2, random_state=42)
rf_metaml.fit(X_train_filled, y_train)
rf_metaml_auc = roc_auc_score(y_test, rf_metaml.predict_proba(X_test_filled), multi_class="ovr", average="macro")

print(f"MetAML-style Random Forest — macro-AUC: {rf_metaml_auc:.3f}")

MetAML-style Random Forest — macro-AUC: 0.878


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

X_train_scaled = StandardScaler().fit_transform(X_train_filled)
X_test_scaled = StandardScaler().fit(X_train_filled).transform(X_test_filled)

# MetAML's alpha grid: np.logspace(-4, -0.5, 50) converted to C values (C = 1/alpha)
C_grid = list(1 / np.logspace(-4, -0.5, 20))  # reduced to 20 points for speed

lasso_metaml = LogisticRegression(penalty="l1", solver="liblinear", max_iter=5000, random_state=42)
lasso_grid = GridSearchCV(lasso_metaml, {"C": C_grid}, cv=5, scoring="roc_auc_ovr", n_jobs=-1)
lasso_grid.fit(X_train_scaled, y_train)
lasso_auc = roc_auc_score(y_test, lasso_grid.predict_proba(X_test_scaled), multi_class="ovr", average="macro")

# MetAML's Elastic Net grid: same alpha grid + l1_ratio [0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0]
enet_metaml = LogisticRegression(penalty="elasticnet", solver="saga", max_iter=10000, random_state=42)
enet_grid = GridSearchCV(enet_metaml, {"C": C_grid[:5], "l1_ratio": [0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0]}, cv=5, scoring="roc_auc_ovr", n_jobs=-1)
enet_grid.fit(X_train_scaled, y_train)
enet_auc = roc_auc_score(y_test, enet_grid.predict_proba(X_test_scaled), multi_class="ovr", average="macro")

print(f"MetAML-style LASSO — macro-AUC: {lasso_auc:.3f}")
print(f"MetAML-style Elastic Net — macro-AUC: {enet_auc:.3f}")

MetAML-style LASSO — macro-AUC: 0.874
MetAML-style Elastic Net — macro-AUC: 0.892


In [5]:
from sklearn.svm import SVC

svm_metaml = SVC(probability=True, random_state=42)
svm_grid = GridSearchCV(svm_metaml, {"C": [0.1, 1, 10, 100], "gamma": ["scale", "auto"]}, cv=5, scoring="roc_auc_ovr", n_jobs=-1)
svm_grid.fit(X_train_scaled, y_train)
svm_auc = roc_auc_score(y_test, svm_grid.predict_proba(X_test_scaled), multi_class="ovr", average="macro")

print(f"MetAML-style SVM — macro-AUC: {svm_auc:.3f}")

MetAML-style SVM — macro-AUC: 0.886
